# ETL Pipeline: Livestock Intelligence — FASE 2 (TRANSFORM)
**Laporan Praktikum Teknologi Perekayasaan Data — Kelompok 1 (3SI1)**

---

## Scope Transform

Fase ini mengambil data **as-is** dari *Staging Area* (`staging_db` PostgreSQL) yang telah diisi pada Fase Extract, kemudian melakukan seluruh proses transformasi menggunakan **PySpark** hingga menghasilkan tabel fakta `fact_supply_resilience` yang siap di-load ke *Data Warehouse*.

### Alur Transform:
```
staging_db (PostgreSQL)
    │
    ├── [LANGKAH 1] Cleaning & Merge BPS (API + Dummy)
    ├── [LANGKAH 2] Normalisasi & Unpivot → 4 tabel (ref_wilayah, ref_komoditas, tr_demografi, tr_statistik)
    ├── [LANGKAH 3] Preprocessing Lanjutan (Standardisasi provinsi, Harmonisasi tanggal, Imputasi)
    ├── [LANGKAH 4] Bangun Tabel Dimensi (dim_prov, dim_komoditas, dim_waktu)
    ├── [LANGKAH 5A] Agregasi iSIKHNAS & PIHPS
    ├── [LANGKAH 5B] Normalisasi & Konversi Data BPS
    ├── [LANGKAH 5C] Integrasi (INNER JOIN Agregasi + BPS)
    └── [LANGKAH 5D] Hitung supply_risk_index → fact_supply_resilience
```

In [1]:
import os
# SET DULU (HARUS PALING ATAS)
os.environ["PYSPARK_PYTHON"] = r"C:\laragon\bin\python\python-3.10\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\laragon\bin\python\python-3.10\python.exe"
os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["PATH"] += ";C:/hadoop/bin"
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, LongType, StringType, DateType, TimestampType
)
from pyspark.sql.window import Window
PG_USER     = 'postgres'
PG_PASS     = '-RqorROOT44'
PG_HOST     = 'localhost'
PG_PORT     = '5432'
STAGING_DB  = 'staging_db'

PG_JDBC_URL  = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{STAGING_DB}"
PG_JDBC_PROPS = {
    "user":     PG_USER,
    "password": PG_PASS,
    "driver":   "org.postgresql.Driver"
}

# ── INISIALISASI SPARK SESSION ──────────────────────────
# Pastikan postgresql JDBC driver tersedia.

JDBC_JAR = "D:/STIS SEM 6/TPD/TPD UTS KELOMPOK 1/postgresql-42.7.11.jar"


try:
    spark.stop()
except:
    pass

spark = (SparkSession.builder
    .appName("Livestock Intelligence — Transform")
    .config("spark.jars", JDBC_JAR)
    .config("spark.driver.extraClassPath", JDBC_JAR)
    .config("spark.executor.extraClassPath", JDBC_JAR)
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
    .config("spark.hadoop.io.native.lib.available", "false")    
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession berhasil diinisialisasi.")
print(f"   Spark version : {spark.version}")
print(f"   JDBC URL      : {PG_JDBC_URL}")

✅ SparkSession berhasil diinisialisasi.
   Spark version : 4.1.1
   JDBC URL      : jdbc:postgresql://localhost:5432/staging_db


In [2]:
print(spark.sparkContext._conf.get("spark.jars"))

D:/STIS SEM 6/TPD/TPD UTS KELOMPOK 1/postgresql-42.7.11.jar


In [3]:
# =========================================================
# CELL 2 · HELPER FUNCTIONS
# =========================================================

def read_staging(table_name: str):
    """Membaca tabel dari staging_db PostgreSQL via JDBC."""
    df = (spark.read
          .format("jdbc")
          .option("url",      PG_JDBC_URL)
          .option("dbtable",  f"public.{table_name}")
          .option("user",     PG_USER)
          .option("password", PG_PASS)
          .option("driver",   "org.postgresql.Driver")
          .load())
    print(f"  📥 {table_name:<45} → {df.count():>6} baris, {len(df.columns)} kolom")
    return df

def show_info(df, label: str, n: int = 5):
    """Menampilkan info singkat dataframe."""
    print(f"\n{'='*60}")
    print(f"  {label}  |  {df.count()} baris × {len(df.columns)} kolom")
    print(f"{'='*60}")
    df.printSchema()
    df.show(n, truncate=False)

print("✅ Helper functions siap.")

✅ Helper functions siap.


In [4]:
# =========================================================
# CELL 3 · LOAD SEMUA TABEL DARI STAGING_DB
# =========================================================
print("🔄 Memuat tabel staging dari PostgreSQL...\n")

# ── BPS ─────────────────────────────────────────────────
sdf_bps_api   = read_staging("staging_bps_api_raw")
sdf_bps_dummy = read_staging("staging_bps_dummy_raw")

# ── iSIKHNAS ────────────────────────────────────────────
sdf_ref_hewan    = read_staging("staging_isikhnas_ref_hewan")
sdf_ref_wilayah  = read_staging("staging_isikhnas_ref_wilayah")
sdf_mutasi       = read_staging("staging_isikhnas_tr_mutasi")
sdf_laporan      = read_staging("staging_isikhnas_tr_laporan_sakit")
sdf_lab          = read_staging("staging_isikhnas_tr_hasil_lab")
sdf_rph          = read_staging("staging_isikhnas_tr_rph")

# ── PIHPS ────────────────────────────────────────────────
sdf_pihps        = read_staging("staging_pihps_raw")

print("\n✅ Semua tabel staging berhasil dimuat.")

🔄 Memuat tabel staging dari PostgreSQL...

  📥 staging_bps_api_raw                           →    224 baris, 6 kolom
  📥 staging_bps_dummy_raw                         →    228 baris, 16 kolom
  📥 staging_isikhnas_ref_hewan                    →      2 baris, 3 kolom
  📥 staging_isikhnas_ref_wilayah                  →     38 baris, 2 kolom
  📥 staging_isikhnas_tr_mutasi                    →    500 baris, 6 kolom
  📥 staging_isikhnas_tr_laporan_sakit             →    300 baris, 6 kolom
  📥 staging_isikhnas_tr_hasil_lab                 →    300 baris, 5 kolom
  📥 staging_isikhnas_tr_rph                       →    400 baris, 5 kolom
  📥 staging_pihps_raw                             →  21966 baris, 6 kolom

✅ Semua tabel staging berhasil dimuat.


---
## LANGKAH 1 — Cleaning & Merge Data BPS

Data BPS API masuk sebagai teks mentah. Perlu membersihkan format angka (titik ribuan, koma desimal) dan mengonversi ke tipe numerik sebelum bisa di-merge.

In [5]:
# =========================================================
# CELL 4 · LANGKAH 1A — CLEANING BPS API RAW
# =========================================================
# Kolom dari staging_bps_api_raw masih TEXT semua.
# Format angka BPS: titik (.) = pemisah ribuan, koma (,) = desimal

def clean_bps_num(df, col_name: str):
    """Hapus titik ribuan, ganti koma desimal → cast float."""
    return (df
        .withColumn(col_name,
            F.regexp_replace(F.col(col_name), r'\.', '')   # hapus titik ribuan
        )
        .withColumn(col_name,
            F.regexp_replace(F.col(col_name), ',', '.')     # ganti koma → titik
        )
        .withColumn(col_name, F.col(col_name).cast(DoubleType()))
    )

numeric_api_cols = [
    "jumlah_penduduk",
    "populasi_sapi",
    "populasi_ayam",
    "produksi_daging_ayam"
]

sdf_bps_api_clean = sdf_bps_api
for c in numeric_api_cols:
    sdf_bps_api_clean = clean_bps_num(sdf_bps_api_clean, c)

# Cast tahun
sdf_bps_api_clean = sdf_bps_api_clean.withColumn("tahun", F.col("tahun").cast(IntegerType()))

print("✅ Cleaning BPS API selesai.")
show_info(sdf_bps_api_clean, "BPS API — setelah cleaning")

✅ Cleaning BPS API selesai.

  BPS API — setelah cleaning  |  224 baris × 6 kolom
root
 |-- provinsi: string (nullable = true)
 |-- tahun: integer (nullable = true)
 |-- jumlah_penduduk: double (nullable = true)
 |-- populasi_sapi: double (nullable = true)
 |-- populasi_ayam: double (nullable = true)
 |-- produksi_daging_ayam: double (nullable = true)

+--------+-----+---------------+-------------+-------------+--------------------+
|provinsi|tahun|jumlah_penduduk|populasi_sapi|populasi_ayam|produksi_daging_ayam|
+--------+-----+---------------+-------------+-------------+--------------------+
|Aceh    |2020 |5274.9         |435376.0     |3.2590982E7  |3.5935469E7         |
|Aceh    |2021 |5333.7         |455177.0     |3.4075587E7  |3.757242321E7       |
|Aceh    |2022 |5407.9         |533593.0     |3.5674912E7  |4.203139871E7       |
|Aceh    |2023 |5482.5         |229295.0     |3.5758787E7  |4.172639265E7       |
|Aceh    |2024 |5554.8         |461487.0     |3.6541907E7  |4.452172367

In [6]:
# =========================================================
# CELL 5 · LANGKAH 1B — CLEANING BPS DUMMY RAW
# =========================================================
# Dummy sudah bertipe numerik dari staging, tapi perlu standarisasi nama kolom
# agar bisa di-merge (suffix _dummy akan dipertahankan sampai merge selesai).

sdf_bps_dummy_clean = sdf_bps_dummy.withColumn(
    "tahun", F.col("tahun").cast(IntegerType())
)

print("✅ Cleaning BPS Dummy selesai.")
show_info(sdf_bps_dummy_clean, "BPS Dummy — setelah cleaning", n=3)

✅ Cleaning BPS Dummy selesai.

  BPS Dummy — setelah cleaning  |  228 baris × 16 kolom
root
 |-- provinsi: string (nullable = true)
 |-- tahun: integer (nullable = true)
 |-- jumlah_penduduk_dummy: long (nullable = true)
 |-- populasi_sapi_dummy: long (nullable = true)
 |-- populasi_ayam_dummy: long (nullable = true)
 |-- produksi_daging_sapi_dummy: double (nullable = true)
 |-- produksi_daging_ayam_dummy: double (nullable = true)
 |-- konsumsi_daging_sapi_dummy: double (nullable = true)
 |-- konsumsi_daging_ayam_dummy: double (nullable = true)
 |-- permintaan_daging_sapi_dummy: double (nullable = true)
 |-- permintaan_daging_ayam_dummy: double (nullable = true)
 |-- jumlah_ternak_sapi_potong_dummy: long (nullable = true)
 |-- jumlah_ternak_ayam_potong_dummy: long (nullable = true)
 |-- harga_baseline_sapi_dummy: long (nullable = true)
 |-- harga_baseline_ayam_dummy: long (nullable = true)
 |-- growth_populasi_dummy: double (nullable = true)

+--------+-----+---------------------+-----

In [7]:
# =========================================================
# CELL 6 · LANGKAH 1C — MERGE BPS API + DUMMY
# =========================================================
# JOIN pada (provinsi, tahun).
# Kolom dari API menjadi sumber utama; kolom tambahan diambil dari dummy.

sdf_bps_master = (sdf_bps_api_clean.alias("api")
    .join(
        sdf_bps_dummy_clean.alias("dum"),
        on=["provinsi", "tahun"],
        how="inner"
    )
    # Pilih & rename kolom final
    .select(
        F.col("api.provinsi"),
        F.col("api.tahun"),
        F.col("api.jumlah_penduduk").alias("jumlah_penduduk"),
        F.col("api.populasi_sapi").alias("populasi_sapi"),
        F.col("api.populasi_ayam").alias("populasi_ayam"),
        F.col("api.produksi_daging_ayam").alias("produksi_daging_ayam"),
        # Kolom eksklusif dari dummy
        F.col("dum.populasi_sapi_dummy"),
        F.col("dum.populasi_ayam_dummy"),
        F.col("dum.produksi_daging_sapi_dummy").alias("produksi_daging_sapi"),
        F.col("dum.produksi_daging_ayam_dummy"),
        F.col("dum.konsumsi_daging_sapi_dummy").alias("konsumsi_daging_sapi"),
        F.col("dum.konsumsi_daging_ayam_dummy").alias("konsumsi_daging_ayam"),
        F.col("dum.permintaan_daging_sapi_dummy").alias("permintaan_daging_sapi"),
        F.col("dum.permintaan_daging_ayam_dummy").alias("permintaan_daging_ayam"),
        F.col("dum.jumlah_ternak_sapi_potong_dummy").alias("jumlah_ternak_sapi_potong"),
        F.col("dum.jumlah_ternak_ayam_potong_dummy").alias("jumlah_ternak_ayam_potong"),
        F.col("dum.harga_baseline_sapi_dummy").alias("harga_baseline_sapi"),
        F.col("dum.harga_baseline_ayam_dummy").alias("harga_baseline_ayam"),
        F.col("dum.growth_populasi_dummy").alias("growth_populasi"),
    )
)

sdf_bps_master.cache()
print(f"✅ BPS Master setelah merge: {sdf_bps_master.count()} baris")
sdf_bps_master.show(5, truncate=False)

✅ BPS Master setelah merge: 218 baris
+-------------------+-----+---------------+-------------+-------------+--------------------+-------------------+-------------------+--------------------+--------------------------+--------------------+--------------------+----------------------+----------------------+-------------------------+-------------------------+-------------------+-------------------+---------------+
|provinsi           |tahun|jumlah_penduduk|populasi_sapi|populasi_ayam|produksi_daging_ayam|populasi_sapi_dummy|populasi_ayam_dummy|produksi_daging_sapi|produksi_daging_ayam_dummy|konsumsi_daging_sapi|konsumsi_daging_ayam|permintaan_daging_sapi|permintaan_daging_ayam|jumlah_ternak_sapi_potong|jumlah_ternak_ayam_potong|harga_baseline_sapi|harga_baseline_ayam|growth_populasi|
+-------------------+-----+---------------+-------------+-------------+--------------------+-------------------+-------------------+--------------------+--------------------------+--------------------+-------

---
## LANGKAH 2 — Normalisasi Tabel BPS & Unpivot

BPS master dipecah menjadi 4 tabel:
| Tabel | Isi |
|---|---|
| `ref_wilayah` | ID & nama provinsi |
| `ref_komoditas` | ID & nama komoditas (Sapi, Ayam) |
| `tr_demografi` | Penduduk & growth per wilayah-tahun |
| `tr_statistik` | Populasi & produksi per wilayah-tahun-komoditas (hasil unpivot) |

In [8]:
# =========================================================
# CELL 7 · LANGKAH 2A — TABEL ref_wilayah (dari BPS master)
# =========================================================
# DISTINCT provinsi, diurutkan alfabetis, beri id_wilayah berurutan.

window_prov = Window.orderBy("provinsi")

sdf_ref_wilayah_bps = (sdf_bps_master
    .select("provinsi").distinct()
    .orderBy("provinsi")
    .withColumn("id_wilayah", F.row_number().over(window_prov))
    .select("id_wilayah", "provinsi")
)

sdf_ref_wilayah_bps.cache()
print(f"✅ ref_wilayah BPS: {sdf_ref_wilayah_bps.count()} provinsi")
sdf_ref_wilayah_bps.show(40, truncate=False)

✅ ref_wilayah BPS: 37 provinsi
+----------+-------------------+
|id_wilayah|provinsi           |
+----------+-------------------+
|1         |Aceh               |
|2         |Bali               |
|3         |Banten             |
|4         |Bengkulu           |
|5         |DI Yogyakarta      |
|6         |DKI Jakarta        |
|7         |Gorontalo          |
|8         |Jambi              |
|9         |Jawa Barat         |
|10        |Jawa Tengah        |
|11        |Jawa Timur         |
|12        |Kalimantan Barat   |
|13        |Kalimantan Selatan |
|14        |Kalimantan Tengah  |
|15        |Kalimantan Timur   |
|16        |Kalimantan Utara   |
|17        |Kepulauan Riau     |
|18        |Lampung            |
|19        |Maluku             |
|20        |Maluku Utara       |
|21        |Nusa Tenggara Barat|
|22        |Nusa Tenggara Timur|
|23        |Papua              |
|24        |Papua Barat        |
|25        |Papua Barat Daya   |
|26        |Papua Pegunungan   |
|27        |

In [9]:
# =========================================================
# CELL 8 · LANGKAH 2B — TABEL ref_komoditas
# =========================================================
from pyspark.sql.types import StructType, StructField

ref_komoditas_data = [(1, "Sapi"), (2, "Ayam")]
sdf_ref_komoditas = spark.createDataFrame(ref_komoditas_data, ["id_komoditas", "nama_komoditas"])

print("✅ ref_komoditas:")
sdf_ref_komoditas.show()

✅ ref_komoditas:
+------------+--------------+
|id_komoditas|nama_komoditas|
+------------+--------------+
|           1|          Sapi|
|           2|          Ayam|
+------------+--------------+



In [10]:
# =========================================================
# CELL 9 · LANGKAH 2C — MAPPING id_wilayah ke BPS Master
# =========================================================
# JOIN tabel utama dengan ref_wilayah → inject id_wilayah → drop kolom teks provinsi

sdf_bps_mapped = (sdf_bps_master.alias("bps")
    .join(sdf_ref_wilayah_bps.alias("ref"), on="provinsi", how="left")
    .drop("provinsi")   # Dari titik ini mesin hanya berurusan dengan angka
)

print(f"✅ BPS Master setelah mapping: {sdf_bps_mapped.count()} baris")
print("Kolom:", sdf_bps_mapped.columns)

✅ BPS Master setelah mapping: 218 baris
Kolom: ['tahun', 'jumlah_penduduk', 'populasi_sapi', 'populasi_ayam', 'produksi_daging_ayam', 'populasi_sapi_dummy', 'populasi_ayam_dummy', 'produksi_daging_sapi', 'produksi_daging_ayam_dummy', 'konsumsi_daging_sapi', 'konsumsi_daging_ayam', 'permintaan_daging_sapi', 'permintaan_daging_ayam', 'jumlah_ternak_sapi_potong', 'jumlah_ternak_ayam_potong', 'harga_baseline_sapi', 'harga_baseline_ayam', 'growth_populasi', 'id_wilayah']


In [11]:
# =========================================================
# CELL 10 · LANGKAH 2D — TABEL tr_demografi
# =========================================================
# Subset kolom kependudukan. Buang duplikasi (karena 1 baris sudah unik per
# wilayah-tahun pada BPS master).

sdf_tr_demografi = (sdf_bps_mapped
    .select("id_wilayah", "tahun", "jumlah_penduduk", "growth_populasi")
    .dropDuplicates(["id_wilayah", "tahun"])
)

sdf_tr_demografi.cache()
print(f"✅ tr_demografi: {sdf_tr_demografi.count()} baris")
sdf_tr_demografi.show(5)

✅ tr_demografi: 218 baris
+----------+-----+---------------+---------------+
|id_wilayah|tahun|jumlah_penduduk|growth_populasi|
+----------+-----+---------------+---------------+
|        36| 2023|         8743.5|        -0.0163|
|        11| 2023|        41527.9|          0.027|
|        26| 2021|           NULL|         0.0183|
|         2| 2025|         4461.3|         0.0187|
|        30| 2025|         1525.3|         0.0386|
+----------+-----+---------------+---------------+
only showing top 5 rows


In [12]:
# =========================================================
# CELL 11 · LANGKAH 2E — TABEL tr_statistik (UNPIVOT / MELT)
# =========================================================
# Unpivot kolom populasi & produksi dari wide → long per komoditas.

# ── SAPI ──
sdf_sapi = (sdf_bps_mapped.select(
    "id_wilayah", "tahun",
    F.lit(1).alias("id_komoditas"),
    F.coalesce(F.col("populasi_sapi"), F.col("populasi_sapi_dummy")).alias("populasi"),
    F.col("produksi_daging_sapi").alias("produksi_daging"),
    F.col("konsumsi_daging_sapi").alias("konsumsi_daging"),
    F.col("permintaan_daging_sapi").alias("permintaan_daging"),
    F.col("jumlah_ternak_sapi_potong").alias("jumlah_ternak_potong"),
    F.col("harga_baseline_sapi").cast(DoubleType()).alias("harga_baseline"),
))

# ── AYAM ──
sdf_ayam = (sdf_bps_mapped.select(
    "id_wilayah", "tahun",
    F.lit(2).alias("id_komoditas"),
    F.coalesce(F.col("populasi_ayam"), F.col("populasi_ayam_dummy")).alias("populasi"),
    F.coalesce(F.col("produksi_daging_ayam"), F.col("produksi_daging_ayam_dummy")).alias("produksi_daging"),
    F.col("konsumsi_daging_ayam").alias("konsumsi_daging"),
    F.col("permintaan_daging_ayam").alias("permintaan_daging"),
    F.col("jumlah_ternak_ayam_potong").alias("jumlah_ternak_potong"),
    F.col("harga_baseline_ayam").cast(DoubleType()).alias("harga_baseline"),
))

sdf_tr_statistik = sdf_sapi.unionByName(sdf_ayam)
sdf_tr_statistik.cache()
print(f"✅ tr_statistik (unpivot): {sdf_tr_statistik.count()} baris")
sdf_tr_statistik.show(5)

✅ tr_statistik (unpivot): 436 baris
+----------+-----+------------+---------+---------------+---------------+-----------------+--------------------+--------------+
|id_wilayah|tahun|id_komoditas| populasi|produksi_daging|konsumsi_daging|permintaan_daging|jumlah_ternak_potong|harga_baseline|
+----------+-----+------------+---------+---------------+---------------+-----------------+--------------------+--------------+
|         6| 2024|           1|   2684.0|       22118.43|            2.8|        107380.61|              104655|      109127.0|
|        10| 2023|           1|1213744.0|       57671.64|            1.6|         38554.99|              214728|      122553.0|
|        13| 2020|           1| 148026.0|       14462.43|           1.74|         38294.64|               57442|      119246.0|
|        25| 2021|           1|  97493.0|       12618.43|           1.73|         33081.46|               43598|       90168.0|
|        21| 2022|           1|1219784.0|       60306.76|           

---
## LANGKAH 3 — Preprocessing Lanjutan

Tiga proses utama:
1. **Standardisasi nama provinsi** — nama resmi BPS sebagai standar (diterapkan di tabel iSIKHNAS)
2. **Harmonisasi format tanggal** — semua tanggal ke `DATE` format `YYYY-MM-DD`, ekstrak bulan & tahun
3. **Imputasi missing value** — median untuk harga (PIHPS), mean untuk populasi/produksi (BPS)

In [13]:
# =========================================================
# CELL 12 · LANGKAH 3A — STANDARDISASI NAMA PROVINSI (iSIKHNAS)
# =========================================================
# iSIKHNAS menggunakan id_wilayah integer (1–38) yang sudah sesuai
# dengan ref_wilayah BPS. Kita perlu memastikan mapping-nya konsisten.

# Tabel ref_wilayah iSIKHNAS dari staging (38 provinsi)
sdf_ref_wilayah_isikhnas = sdf_ref_wilayah.select(
    F.col("id_wilayah").cast(IntegerType()),
    F.col("provinsi")
)

# Tabel standar: gunakan nama dari BPS sebagai master
# Karena urutan id_wilayah pada iSIKHNAS sama dengan BPS,
# kita cukup join on id_wilayah untuk mendapatkan nama resmi.
sdf_wilayah_master = (sdf_ref_wilayah_bps.alias("bps_ref")
    .join(sdf_ref_wilayah_isikhnas.alias("isk_ref"),
          F.col("bps_ref.id_wilayah") == F.col("isk_ref.id_wilayah"),
          how="full")
    .select(
        F.coalesce(F.col("bps_ref.id_wilayah"), F.col("isk_ref.id_wilayah")).alias("id_wilayah"),
        F.coalesce(F.col("bps_ref.provinsi"), F.col("isk_ref.provinsi")).alias("provinsi")
    )
    .orderBy("id_wilayah")
)

sdf_wilayah_master.cache()
print(f"✅ wilayah_master: {sdf_wilayah_master.count()} entri")
sdf_wilayah_master.show(40, truncate=False)

✅ wilayah_master: 38 entri
+----------+-------------------+
|id_wilayah|provinsi           |
+----------+-------------------+
|1         |Aceh               |
|2         |Bali               |
|3         |Banten             |
|4         |Bengkulu           |
|5         |DI Yogyakarta      |
|6         |DKI Jakarta        |
|7         |Gorontalo          |
|8         |Jambi              |
|9         |Jawa Barat         |
|10        |Jawa Tengah        |
|11        |Jawa Timur         |
|12        |Kalimantan Barat   |
|13        |Kalimantan Selatan |
|14        |Kalimantan Tengah  |
|15        |Kalimantan Timur   |
|16        |Kalimantan Utara   |
|17        |Kepulauan Riau     |
|18        |Lampung            |
|19        |Maluku             |
|20        |Maluku Utara       |
|21        |Nusa Tenggara Barat|
|22        |Nusa Tenggara Timur|
|23        |Papua              |
|24        |Papua Barat        |
|25        |Papua Barat Daya   |
|26        |Papua Pegunungan   |
|27        |Papu

In [14]:
# =========================================================
# CELL 13 · LANGKAH 3B — HARMONISASI FORMAT TANGGAL (iSIKHNAS)
# =========================================================
# Tanggal masuk sebagai string format MM/dd/yy atau MM/dd/yyyy.
# Target: DateType YYYY-MM-DD + kolom bulan & tahun diekstrak.

def parse_isikhnas_date(df, col_in: str, col_out: str = None):
    """Parse tanggal iSIKHNAS MM/dd/yy → DATE, tambah kolom bulan & tahun."""
    out = col_out or col_in
    return (df
        .withColumn(f"_{out}_str",
            # Coba format dua digit tahun dulu, fallback empat digit
            F.to_date(F.col(col_in), "MM/dd/yy")
        )
        .withColumn(f"_{out}_str4",
            F.to_date(F.col(col_in), "MM/dd/yyyy")
        )
        .withColumn(out,
            F.coalesce(F.col(f"_{out}_str"), F.col(f"_{out}_str4"))
        )
        .drop(f"_{out}_str", f"_{out}_str4", col_in if col_in != out else "__dummy")
        .withColumn(f"{out}_bulan", F.month(F.col(out)).cast(IntegerType()))
        .withColumn(f"{out}_tahun", F.year(F.col(out)).cast(IntegerType()))
    )

# ── Laporan Sakit ──
sdf_laporan_clean = parse_isikhnas_date(sdf_laporan, "tgl_lapor", "tgl_lapor")
sdf_laporan_clean = (sdf_laporan_clean
    .withColumn("id_wilayah", F.col("id_wilayah").cast(IntegerType()))
    .withColumn("jumlah_gejala", F.col("jumlah_gejala").cast(DoubleType()))
    .withColumn("jumlah_mati",   F.col("jumlah_mati").cast(DoubleType()))
)

# ── Mutasi ──
sdf_mutasi_clean = parse_isikhnas_date(sdf_mutasi, "tgl_mutasi", "tgl_mutasi")
sdf_mutasi_clean = (sdf_mutasi_clean
    .withColumn("id_asal",   F.col("id_asal").cast(IntegerType()))
    .withColumn("id_tujuan", F.col("id_tujuan").cast(IntegerType()))
    .withColumn("jumlah_ekor", F.col("jumlah_ekor").cast(DoubleType()))
)

# ── RPH ──
sdf_rph_clean = parse_isikhnas_date(sdf_rph, "tgl_potong", "tgl_potong")
sdf_rph_clean = (sdf_rph_clean
    .withColumn("id_wilayah", F.col("id_wilayah").cast(IntegerType()))
    .withColumn("berat_karkas", F.col("berat_karkas").cast(DoubleType()))
)

# ── Lab ──
sdf_lab_clean = parse_isikhnas_date(sdf_lab, "tgl_uji", "tgl_uji")

print("✅ Harmonisasi tanggal selesai.")
print("\nSampel laporan_sakit:")
sdf_laporan_clean.show(3, truncate=False)
print("\nSampel rph:")
sdf_rph_clean.show(3, truncate=False)

✅ Harmonisasi tanggal selesai.

Sampel laporan_sakit:
+----------+----------+----------+--------+-------------+-----------+---------------+---------------+
|id_laporan|tgl_lapor |id_wilayah|id_hewan|jumlah_gejala|jumlah_mati|tgl_lapor_bulan|tgl_lapor_tahun|
+----------+----------+----------+--------+-------------+-----------+---------------+---------------+
|LAP-0001  |2025-11-10|18        |H2-001  |21.0         |4.0        |11             |2025           |
|LAP-0002  |2024-12-19|27        |H4-001  |NULL         |NULL       |12             |2024           |
|LAP-0003  |2024-01-24|14        |H2-001  |31.0         |4.0        |1              |2024           |
+----------+----------+----------+--------+-------------+-----------+---------------+---------------+
only showing top 3 rows

Sampel rph:
+---------+----------+----------+--------+------------+----------------+----------------+
|id_potong|tgl_potong|id_wilayah|id_hewan|berat_karkas|tgl_potong_bulan|tgl_potong_tahun|
+---------+----

In [15]:
# =========================================================
# CELL 14 · LANGKAH 3C — HARMONISASI TANGGAL PIHPS
# =========================================================
# staging_pihps_raw: kolom 'waktu' bertipe TIMESTAMP.

sdf_pihps_clean = (sdf_pihps
    .withColumn("waktu",  F.col("waktu").cast(TimestampType()))
    .withColumn("tgl",    F.to_date(F.col("waktu")))
    .withColumn("bulan",  F.month(F.col("waktu")).cast(IntegerType()))
    .withColumn("tahun",  F.year(F.col("waktu")).cast(IntegerType()))
    .withColumn("harga",  F.col("harga").cast(DoubleType()))
    # Standardisasi nama komoditas ke format Title Case
    .withColumn("nama_komoditas",
        F.when(F.lower(F.col("nama_komoditas")).contains("sapi"), F.lit("Sapi"))
         .when(F.lower(F.col("nama_komoditas")).contains("ayam"), F.lit("Ayam"))
         .otherwise(F.col("nama_komoditas"))
    )
)

sdf_pihps_clean.cache()
print(f"✅ PIHPS clean: {sdf_pihps_clean.count()} baris")
sdf_pihps_clean.printSchema()
sdf_pihps_clean.show(5, truncate=False)

✅ PIHPS clean: 21966 baris
root
 |-- provinsi: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_komoditas: string (nullable = true)
 |-- jenis_pasar: string (nullable = true)
 |-- harga: double (nullable = true)
 |-- waktu: timestamp (nullable = true)
 |-- tgl: date (nullable = true)
 |-- bulan: integer (nullable = true)
 |-- tahun: integer (nullable = true)

+--------+-------+--------------+--------------+--------+-------------------+----------+-----+-----+
|provinsi|kota   |nama_komoditas|jenis_pasar   |harga   |waktu              |tgl       |bulan|tahun|
+--------+-------+--------------+--------------+--------+-------------------+----------+-----+-----+
|NTB     |MATARAM|Ayam          |PEDAGANG BESAR|21750.0 |2024-01-01 00:00:00|2024-01-01|1    |2024 |
|NTB     |MATARAM|Sapi          |PEDAGANG BESAR|110500.0|2024-01-01 00:00:00|2024-01-01|1    |2024 |
|NTB     |MATARAM|Ayam          |PEDAGANG BESAR|21750.0 |2024-01-02 00:00:00|2024-01-02|1    |2024 |
|NTB     |

In [16]:
# =========================================================
# CELL 15 · LANGKAH 3D — IMPUTASI MISSING VALUE
# =========================================================

# ── A. PIHPS: Imputasi harga dengan MEDIAN per komoditas-provinsi-bulan ──
# Hitung median per grup
sdf_pihps_median = (sdf_pihps_clean
    .groupBy("provinsi", "nama_komoditas", "bulan")
    .agg(F.percentile_approx("harga", 0.5).alias("median_harga"))
)

sdf_pihps_imputed = (sdf_pihps_clean.alias("p")
    .join(sdf_pihps_median.alias("m"),
          on=["provinsi", "nama_komoditas", "bulan"],
          how="left")
    .withColumn("harga",
        F.when(F.col("p.harga").isNull(), F.col("m.median_harga"))
         .otherwise(F.col("p.harga"))
    )
    .drop("median_harga")
)

# ── B. BPS tr_statistik: Imputasi populasi & produksi dengan MEAN per wilayah-komoditas ──
sdf_tr_stat_mean = (sdf_tr_statistik
    .groupBy("id_wilayah", "id_komoditas")
    .agg(
        F.mean("populasi").alias("mean_populasi"),
        F.mean("produksi_daging").alias("mean_produksi"),
        F.mean("konsumsi_daging").alias("mean_konsumsi"),
        F.mean("permintaan_daging").alias("mean_permintaan"),
        F.mean("jumlah_ternak_potong").alias("mean_ternak_potong"),
    )
)

sdf_tr_statistik_imputed = (sdf_tr_statistik.alias("s")
    .join(sdf_tr_stat_mean.alias("m"),
          on=["id_wilayah", "id_komoditas"],
          how="left")
    .withColumn("populasi",        F.coalesce(F.col("s.populasi"),        F.col("m.mean_populasi")))
    .withColumn("produksi_daging", F.coalesce(F.col("s.produksi_daging"), F.col("m.mean_produksi")))
    .withColumn("konsumsi_daging", F.coalesce(F.col("s.konsumsi_daging"), F.col("m.mean_konsumsi")))
    .withColumn("permintaan_daging",   F.coalesce(F.col("s.permintaan_daging"),   F.col("m.mean_permintaan")))
    .withColumn("jumlah_ternak_potong",F.coalesce(F.col("s.jumlah_ternak_potong"),F.col("m.mean_ternak_potong")))
    .select("s.id_wilayah","s.tahun","s.id_komoditas",
            "populasi","produksi_daging","konsumsi_daging",
            "permintaan_daging","jumlah_ternak_potong","s.harga_baseline")
)

sdf_tr_statistik_imputed.cache()
print("✅ Imputasi selesai.")
print(f"   PIHPS missing setelah imputasi : {sdf_pihps_imputed.filter(F.col('harga').isNull()).count()}")
print(f"   tr_statistik missing populasi  : {sdf_tr_statistik_imputed.filter(F.col('populasi').isNull()).count()}")

✅ Imputasi selesai.
   PIHPS missing setelah imputasi : 0
   tr_statistik missing populasi  : 0


---
## LANGKAH 4 — Membangun Tabel Dimensi (Surrogate Key & DDL Constraint)

Tiga tabel dimensi dibentuk:
- `dim_prov` — provinsi dengan surrogate key & kode BPS
- `dim_komoditas` — Sapi / Ayam
- `dim_waktu` — deret waktu otomatis (bulan, kuartal, tahun)

In [17]:
# =========================================================
# CELL 16 · LANGKAH 4A — dim_prov
# =========================================================
window_dim_prov = Window.orderBy("id_wilayah")

sdf_dim_prov = (sdf_wilayah_master
    .withColumn("prov_key", F.row_number().over(window_dim_prov))
    .withColumn("kode_bps",
        # Kode BPS: id_wilayah × 100000 (format standar 7 digit)
        (F.col("id_wilayah") * 100000).cast(LongType())
    )
    .select(
        F.col("prov_key"),
        F.col("kode_bps"),
        F.col("provinsi").alias("nama_provinsi"),
        F.col("id_wilayah")   # retained for join reference
    )
)

sdf_dim_prov.cache()
print(f"✅ dim_prov: {sdf_dim_prov.count()} baris")
sdf_dim_prov.show(40, truncate=False)

✅ dim_prov: 38 baris
+--------+--------+-------------------+----------+
|prov_key|kode_bps|nama_provinsi      |id_wilayah|
+--------+--------+-------------------+----------+
|1       |100000  |Aceh               |1         |
|2       |200000  |Bali               |2         |
|3       |300000  |Banten             |3         |
|4       |400000  |Bengkulu           |4         |
|5       |500000  |DI Yogyakarta      |5         |
|6       |600000  |DKI Jakarta        |6         |
|7       |700000  |Gorontalo          |7         |
|8       |800000  |Jambi              |8         |
|9       |900000  |Jawa Barat         |9         |
|10      |1000000 |Jawa Tengah        |10        |
|11      |1100000 |Jawa Timur         |11        |
|12      |1200000 |Kalimantan Barat   |12        |
|13      |1300000 |Kalimantan Selatan |13        |
|14      |1400000 |Kalimantan Tengah  |14        |
|15      |1500000 |Kalimantan Timur   |15        |
|16      |1600000 |Kalimantan Utara   |16        |
|17      |

In [18]:
# =========================================================
# CELL 17 · LANGKAH 4B — dim_komoditas
# =========================================================
from pyspark.sql.types import StructType, StructField, IntegerType as IT, StringType as ST

schema_kom = StructType([
    StructField("komoditas_key", IT(), False),
    StructField("id_komoditas",  IT(), False),
    StructField("nama_komoditas",ST(), True),
])
sdf_dim_komoditas = spark.createDataFrame(
    [(1, 1, "Sapi"), (2, 2, "Ayam")],
    schema_kom
)

print("✅ dim_komoditas:")
sdf_dim_komoditas.show()

✅ dim_komoditas:
+-------------+------------+--------------+
|komoditas_key|id_komoditas|nama_komoditas|
+-------------+------------+--------------+
|            1|           1|          Sapi|
|            2|           2|          Ayam|
+-------------+------------+--------------+



In [19]:
# =========================================================
# CELL 18 · LANGKAH 4C — dim_waktu
# =========================================================
# Deret waktu: semua kombinasi bulan-tahun dari rentang data.

# Ambil min-max tahun dari data
tahun_min = sdf_tr_statistik_imputed.agg(F.min("tahun")).collect()[0][0]
tahun_max = sdf_tr_statistik_imputed.agg(F.max("tahun")).collect()[0][0]
print(f"   Rentang tahun: {tahun_min} — {tahun_max}")

# Buat deret bulan-tahun
import itertools
waktu_rows = [
    (y, m) for y in range(tahun_min, tahun_max + 1) for m in range(1, 13)
]

sdf_waktu_base = spark.createDataFrame(waktu_rows, ["tahun", "bulan"])

nama_bulan_map = {
    1:"Januari",2:"Februari",3:"Maret",4:"April",5:"Mei",6:"Juni",
    7:"Juli",8:"Agustus",9:"September",10:"Oktober",11:"November",12:"Desember"
}
nama_bulan_expr = F.create_map([F.lit(x) for pair in nama_bulan_map.items() for x in pair])

window_waktu = Window.orderBy("tahun", "bulan")

sdf_dim_waktu = (sdf_waktu_base
    .withColumn("waktu_key", F.row_number().over(window_waktu))
    .withColumn("nama_bulan", nama_bulan_expr[F.col("bulan")])
    .withColumn("kuartal",
        F.when(F.col("bulan").between(1,3), F.lit("Q1"))
         .when(F.col("bulan").between(4,6), F.lit("Q2"))
         .when(F.col("bulan").between(7,9), F.lit("Q3"))
         .otherwise(F.lit("Q4"))
    )
    .select("waktu_key", "bulan", "nama_bulan", "kuartal", "tahun")
)

sdf_dim_waktu.cache()
print(f"✅ dim_waktu: {sdf_dim_waktu.count()} baris")
sdf_dim_waktu.show(15)

   Rentang tahun: 2020 — 2025
✅ dim_waktu: 72 baris
+---------+-----+----------+-------+-----+
|waktu_key|bulan|nama_bulan|kuartal|tahun|
+---------+-----+----------+-------+-----+
|        1|    1|   Januari|     Q1| 2020|
|        2|    2|  Februari|     Q1| 2020|
|        3|    3|     Maret|     Q1| 2020|
|        4|    4|     April|     Q2| 2020|
|        5|    5|       Mei|     Q2| 2020|
|        6|    6|      Juni|     Q2| 2020|
|        7|    7|      Juli|     Q3| 2020|
|        8|    8|   Agustus|     Q3| 2020|
|        9|    9| September|     Q3| 2020|
|       10|   10|   Oktober|     Q4| 2020|
|       11|   11|  November|     Q4| 2020|
|       12|   12|  Desember|     Q4| 2020|
|       13|    1|   Januari|     Q1| 2021|
|       14|    2|  Februari|     Q1| 2021|
|       15|    3|     Maret|     Q1| 2021|
+---------+-----+----------+-------+-----+
only showing top 15 rows


---
## LANGKAH 5A — Agregasi iSIKHNAS & PIHPS

Agregasi metrik operasional `GROUP BY (prov_key, waktu_key, komoditas_key)` menghasilkan:
- **Metrik iSIKHNAS**: sum_jumlah_sakit, sum_jumlah_mati, sum_vol_mutasi, sum_realisasi_karkas
- **Metrik PIHPS**: avg_harga

In [20]:
# =========================================================
# CELL 19 · LANGKAH 5A-i — AGREGASI iSIKHNAS (Laporan Sakit)
# =========================================================

# Join laporan_sakit ← hewan (untuk id_komoditas)
sdf_laporan_joined = (sdf_laporan_clean.alias("lap")
    .join(sdf_ref_hewan.alias("hw"),
          F.col("lap.id_hewan") == F.col("hw.id_hewan"),
          how="left")
    # Map nama hewan → id_komoditas (Sapi=1, Ayam=2)
    .withColumn("id_komoditas",
        F.when(F.lower(F.col("hw.nama_hewan")) == "sapi", F.lit(1))
         .when(F.lower(F.col("hw.nama_hewan")) == "ayam", F.lit(2))
         .otherwise(F.lit(0))
    )
    .withColumn("id_wilayah", F.col("lap.id_wilayah").cast(IntegerType()))
)

# Agregasi: SUM per wilayah-bulan-komoditas
sdf_agg_sakit = (sdf_laporan_joined
    .groupBy(
        "id_wilayah",
        F.col("tgl_lapor_bulan").alias("bulan"),
        F.col("tgl_lapor_tahun").alias("tahun"),
        "id_komoditas"
    )
    .agg(
        F.sum("jumlah_gejala").alias("sum_jumlah_sakit"),
        F.sum("jumlah_mati").alias("sum_jumlah_mati")
    )
)

print(f"✅ Agregasi laporan_sakit: {sdf_agg_sakit.count()} baris")
sdf_agg_sakit.show(5)

✅ Agregasi laporan_sakit: 277 baris
+----------+-----+-----+------------+----------------+---------------+
|id_wilayah|bulan|tahun|id_komoditas|sum_jumlah_sakit|sum_jumlah_mati|
+----------+-----+-----+------------+----------------+---------------+
|        17|   12| 2025|           1|            29.0|            2.0|
|        18|    5| 2024|           2|             7.0|           11.0|
|        28|    2| 2025|           1|             2.0|           14.0|
|        33|    9| 2025|           2|            38.0|           14.0|
|        28|    1| 2025|           2|            45.0|           20.0|
+----------+-----+-----+------------+----------------+---------------+
only showing top 5 rows


In [21]:
# =========================================================
# CELL 20 · LANGKAH 5A-ii — AGREGASI iSIKHNAS (Mutasi & RPH)
# =========================================================

# ── Mutasi: vol mutasi keluar dari provinsi asal ──
sdf_mutasi_joined = (sdf_mutasi_clean.alias("mut")
    .join(sdf_ref_hewan.alias("hw"),
          F.col("mut.id_hewan") == F.col("hw.id_hewan"), how="left")
    .withColumn("id_komoditas",
        F.when(F.lower(F.col("hw.nama_hewan")) == "sapi", F.lit(1))
         .when(F.lower(F.col("hw.nama_hewan")) == "ayam", F.lit(2))
         .otherwise(F.lit(0))
    )
)

sdf_agg_mutasi = (sdf_mutasi_joined
    .groupBy(
        F.col("id_asal").alias("id_wilayah"),
        F.col("tgl_mutasi_bulan").alias("bulan"),
        F.col("tgl_mutasi_tahun").alias("tahun"),
        "id_komoditas"
    )
    .agg(F.sum("jumlah_ekor").alias("sum_vol_mutasi"))
)

# ── RPH: total berat karkas ──
sdf_rph_joined = (sdf_rph_clean.alias("rph")
    .join(sdf_ref_hewan.alias("hw"),
          F.col("rph.id_hewan") == F.col("hw.id_hewan"), how="left")
    .withColumn("id_komoditas",
        F.when(F.lower(F.col("hw.nama_hewan")) == "sapi", F.lit(1))
         .when(F.lower(F.col("hw.nama_hewan")) == "ayam", F.lit(2))
         .otherwise(F.lit(0))
    )
)

sdf_agg_rph = (sdf_rph_joined
    .groupBy(
        F.col("rph.id_wilayah"),
        F.col("tgl_potong_bulan").alias("bulan"),
        F.col("tgl_potong_tahun").alias("tahun"),
        "id_komoditas"
    )
    .agg(F.sum("berat_karkas").alias("sum_realisasi_karkas"))
)

print(f"✅ Agregasi mutasi : {sdf_agg_mutasi.count()} baris")
print(f"✅ Agregasi RPH    : {sdf_agg_rph.count()} baris")

✅ Agregasi mutasi : 446 baris
✅ Agregasi RPH    : 370 baris


In [22]:
# =========================================================
# CELL 21 · LANGKAH 5A-iii — AGREGASI PIHPS (Rata-rata Harga)
# =========================================================

# Join PIHPS dengan dim_prov (matching pada nama provinsi)
sdf_pihps_prov = (sdf_pihps_imputed.alias("p")
    .join(sdf_dim_prov.alias("dp"),
          F.col("p.provinsi") == F.col("dp.nama_provinsi"),
          how="left")
    .withColumn("id_komoditas",
        F.when(F.col("p.nama_komoditas") == "Sapi", F.lit(1))
         .when(F.col("p.nama_komoditas") == "Ayam", F.lit(2))
         .otherwise(F.lit(0))
    )
)

sdf_agg_harga = (sdf_pihps_prov
    .groupBy(
        F.col("dp.id_wilayah"),
        F.col("p.bulan"),
        F.col("p.tahun"),
        "id_komoditas"
    )
    .agg(F.avg("harga").alias("avg_harga"))
)

print(f"✅ Agregasi PIHPS harga: {sdf_agg_harga.count()} baris")
sdf_agg_harga.show(5)

✅ Agregasi PIHPS harga: 48 baris
+----------+-----+-----+------------+------------------+
|id_wilayah|bulan|tahun|id_komoditas|         avg_harga|
+----------+-----+-----+------------+------------------+
|      NULL|    4| 2025|           1| 138542.6406926407|
|      NULL|    6| 2025|           1|139345.46485260772|
|      NULL|    5| 2024|           2| 40820.18633540373|
|      NULL|    7| 2025|           1|136187.57763975154|
|      NULL|   12| 2025|           2|41492.857142857145|
+----------+-----+-----+------------+------------------+
only showing top 5 rows


In [23]:
# =========================================================
# CELL 22 · LANGKAH 5A-iv — GABUNG SEMUA AGREGASI iSIKHNAS + PIHPS
# =========================================================
# Full outer join sehingga tidak ada data yang hilang.

sdf_agg_isikhnas = (sdf_agg_sakit.alias("sk")
    .join(sdf_agg_mutasi.alias("mut"),
          on=["id_wilayah","bulan","tahun","id_komoditas"], how="full")
    .join(sdf_agg_rph.alias("rph"),
          on=["id_wilayah","bulan","tahun","id_komoditas"], how="full")
    .select(
        F.coalesce(F.col("sk.id_wilayah"),F.col("mut.id_wilayah"),F.col("rph.id_wilayah")).alias("id_wilayah"),
        F.coalesce(F.col("sk.bulan"),F.col("mut.bulan"),F.col("rph.bulan")).alias("bulan"),
        F.coalesce(F.col("sk.tahun"),F.col("mut.tahun"),F.col("rph.tahun")).alias("tahun"),
        F.coalesce(F.col("sk.id_komoditas"),F.col("mut.id_komoditas"),F.col("rph.id_komoditas")).alias("id_komoditas"),
        F.col("sk.sum_jumlah_sakit"),
        F.col("sk.sum_jumlah_mati"),
        F.col("mut.sum_vol_mutasi"),
        F.col("rph.sum_realisasi_karkas"),
    )
)

# Gabung dengan PIHPS
sdf_agg_all = (sdf_agg_isikhnas.alias("isk")
    .join(sdf_agg_harga.alias("ph"),
          on=["id_wilayah","bulan","tahun","id_komoditas"], how="full")
    .select(
        F.coalesce(F.col("isk.id_wilayah"),F.col("ph.id_wilayah")).alias("id_wilayah"),
        F.coalesce(F.col("isk.bulan"),F.col("ph.bulan")).alias("bulan"),
        F.coalesce(F.col("isk.tahun"),F.col("ph.tahun")).alias("tahun"),
        F.coalesce(F.col("isk.id_komoditas"),F.col("ph.id_komoditas")).alias("id_komoditas"),
        F.col("isk.sum_jumlah_sakit"),
        F.col("isk.sum_jumlah_mati"),
        F.col("isk.sum_vol_mutasi"),
        F.col("isk.sum_realisasi_karkas"),
        F.col("ph.avg_harga"),
    )
    .filter(F.col("id_komoditas").isin([1, 2]))   # Hanya Sapi & Ayam
)

sdf_agg_all.cache()
print(f"✅ Total agregasi gabungan: {sdf_agg_all.count()} baris")
sdf_agg_all.show(5)

✅ Total agregasi gabungan: 930 baris
+----------+-----+-----+------------+----------------+---------------+--------------+--------------------+---------+
|id_wilayah|bulan|tahun|id_komoditas|sum_jumlah_sakit|sum_jumlah_mati|sum_vol_mutasi|sum_realisasi_karkas|avg_harga|
+----------+-----+-----+------------+----------------+---------------+--------------+--------------------+---------+
|         4|    9| 2025|           2|            NULL|           NULL|          NULL|               167.5|     NULL|
|        16|    7| 2024|           2|            NULL|           NULL|          NULL|  115.38999999999999|     NULL|
|         5|    1| 2025|           1|            NULL|           NULL|          NULL|                NULL|     NULL|
|        10|    8| 2025|           2|            NULL|           NULL|         198.0|              219.92|     NULL|
|        15|    3| 2025|           1|            NULL|           NULL|          11.0|                NULL|     NULL|
+----------+-----+-----+---

---
## LANGKAH 5B — Normalisasi & Konversi Data BPS

- **Konversi satuan**: produksi daging sapi (ton → kg × 1000)
- **Normalisasi waktu** (bagi 12 → metrik bulanan): avg_produksi, avg_konsumsi, avg_permintaan, avg_pemotongan

In [24]:
# =========================================================
# CELL 23 · LANGKAH 5B — NORMALISASI & KONVERSI BPS
# =========================================================

sdf_bps_bulanan = (sdf_tr_statistik_imputed
    .withColumn("populasi_ternak",         F.col("populasi"))
    # Konversi satuan: Sapi ton→kg, Ayam sudah kg
    .withColumn("produksi_daging_kg",
        F.when(F.col("id_komoditas") == 1,
               F.col("produksi_daging") * F.lit(1000.0))   # ton → kg
         .otherwise(F.col("produksi_daging"))
    )
    # Normalisasi tahunan → bulanan (÷ 12)
    .withColumn("avg_produksi_bulanan",
        (F.col("produksi_daging_kg") / F.lit(12.0)))
    .withColumn("avg_konsumsi_bulanan",
        # konsumsi per kapita × penduduk / 12
        # tr_statistik tidak punya penduduk, join lewat tr_demografi
        (F.col("konsumsi_daging") / F.lit(12.0)))
    .withColumn("avg_permintaan_bulanan",
        (F.col("permintaan_daging") / F.lit(12.0)))
    .withColumn("avg_pemotongan_bulanan",
        (F.col("jumlah_ternak_potong") / F.lit(12.0)))
    .select(
        "id_wilayah", "tahun", "id_komoditas",
        "populasi_ternak", "harga_baseline",
        "avg_produksi_bulanan", "avg_konsumsi_bulanan",
        "avg_permintaan_bulanan", "avg_pemotongan_bulanan"
    )
)

sdf_bps_bulanan.cache()
print(f"✅ BPS bulanan: {sdf_bps_bulanan.count()} baris")
sdf_bps_bulanan.show(5)

✅ BPS bulanan: 436 baris
+----------+-----+------------+---------------+--------------+--------------------+--------------------+----------------------+----------------------+
|id_wilayah|tahun|id_komoditas|populasi_ternak|harga_baseline|avg_produksi_bulanan|avg_konsumsi_bulanan|avg_permintaan_bulanan|avg_pemotongan_bulanan|
+----------+-----+------------+---------------+--------------+--------------------+--------------------+----------------------+----------------------+
|         6| 2024|           1|         2684.0|      109127.0|           1843202.5|  0.2333333333333333|     8948.384166666667|               8721.25|
|        10| 2023|           1|      1213744.0|      122553.0|           4805970.0| 0.13333333333333333|     3212.915833333333|               17894.0|
|        13| 2020|           1|       148026.0|      119246.0|           1205202.5|               0.145|               3191.22|     4786.833333333333|
|        25| 2021|           1|        97493.0|       90168.0|  10515

---
## LANGKAH 5C — Integrasi (INNER JOIN Agregasi + BPS)

Join antara agregasi operasional (per bulan) dengan data BPS (per tahun).
Karena BPS granularitasnya tahunan, nilainya akan terduplikasi di setiap bulan dalam tahun yang sama — ini perilaku yang diharapkan.

In [25]:
# =========================================================
# CELL 24 · LANGKAH 5C — INNER JOIN AGREGASI + BPS
# =========================================================

sdf_integrated = (sdf_agg_all.alias("agg")
    .join(sdf_bps_bulanan.alias("bps"),
          on=[
              F.col("agg.id_wilayah")   == F.col("bps.id_wilayah"),
              F.col("agg.id_komoditas") == F.col("bps.id_komoditas"),
              F.col("agg.tahun")        == F.col("bps.tahun"),
          ],
          how="inner"
    )
    .select(
        F.col("agg.id_wilayah"),
        F.col("agg.bulan"),
        F.col("agg.tahun"),
        F.col("agg.id_komoditas"),
        # Metrik iSIKHNAS
        F.coalesce(F.col("agg.sum_jumlah_sakit"),    F.lit(0.0)).alias("sum_jumlah_sakit"),
        F.coalesce(F.col("agg.sum_jumlah_mati"),     F.lit(0.0)).alias("sum_jumlah_mati"),
        F.coalesce(F.col("agg.sum_vol_mutasi"),      F.lit(0.0)).alias("sum_vol_mutasi"),
        F.coalesce(F.col("agg.sum_realisasi_karkas"),F.lit(0.0)).alias("sum_realisasi_karkas"),
        # Metrik PIHPS (avg_harga bisa null → imputasi di 5D)
        F.col("agg.avg_harga"),
        # Metrik BPS (terduplikasi per bulan dalam tahun yang sama — OK)
        F.col("bps.populasi_ternak"),
        F.col("bps.harga_baseline"),
        F.col("bps.avg_produksi_bulanan"),
        F.col("bps.avg_konsumsi_bulanan"),
        F.col("bps.avg_permintaan_bulanan"),
        F.col("bps.avg_pemotongan_bulanan"),
    )
)

sdf_integrated.cache()
print(f"✅ Data terintegrasi: {sdf_integrated.count()} baris")
sdf_integrated.show(5)

✅ Data terintegrasi: 855 baris
+----------+-----+-----+------------+----------------+---------------+--------------+--------------------+---------+---------------+--------------+--------------------+--------------------+----------------------+----------------------+
|id_wilayah|bulan|tahun|id_komoditas|sum_jumlah_sakit|sum_jumlah_mati|sum_vol_mutasi|sum_realisasi_karkas|avg_harga|populasi_ternak|harga_baseline|avg_produksi_bulanan|avg_konsumsi_bulanan|avg_permintaan_bulanan|avg_pemotongan_bulanan|
+----------+-----+-----+------------+----------------+---------------+--------------+--------------------+---------+---------------+--------------+--------------------+--------------------+----------------------+----------------------+
|         4|    9| 2025|           2|             0.0|            0.0|           0.0|               167.5|     NULL|    1.0699227E7|       26041.0|  1089662.3183333334|  0.9458333333333333|    17740.119166666667|     67098.16666666667|
|        16|    7| 2024| 

---
## LANGKAH 5D — Menghitung `supply_risk_index`

Indeks risiko dihitung dari tiga komponen:

| Komponen | Formula | Bobot |
|---|---|---|
| **Price Gap** | `(avg_harga − harga_baseline) / harga_baseline` | 1/3 |
| **Health Impact** | `(sum_jumlah_sakit + sum_jumlah_mati) / populasi_ternak` | 1/3 |
| **Supply Strain** | `sum_vol_mutasi / avg_permintaan_bulanan` | 1/3 |

Setiap komponen di-**min-max scaling** ke `[0.0, 1.0]` menggunakan `greatest`/`least` di PySpark sebelum dirata-rata.

In [26]:
# =========================================================
# CELL 25 · LANGKAH 5D-i — IMPUTASI avg_harga YANG KOSONG
# =========================================================
# Jika avg_harga kosong pada bulan tertentu, gunakan median provinsi-komoditas
# dari bulan sebelumnya sebagai fallback.

window_median_fallback = Window.partitionBy("id_wilayah","id_komoditas").orderBy("tahun","bulan")

sdf_with_prev_median = (sdf_integrated
    .withColumn("prev_harga_median",
        F.last(F.col("avg_harga"), ignorenulls=True)
         .over(window_median_fallback.rowsBetween(Window.unboundedPreceding, -1))
    )
    .withColumn("avg_harga",
        F.when(F.col("avg_harga").isNull(), F.col("prev_harga_median"))
         .otherwise(F.col("avg_harga"))
    )
    .drop("prev_harga_median")
)

null_after = sdf_with_prev_median.filter(F.col("avg_harga").isNull()).count()
print(f"✅ Imputasi avg_harga selesai. Sisa null: {null_after}")

✅ Imputasi avg_harga selesai. Sisa null: 855


In [27]:
# =========================================================
# CELL 26 · LANGKAH 5D-ii — HITUNG KOMPONEN INDEKS (RAW)
# =========================================================
# Guard division by zero: jika pembagi = 0 atau null → komponen = 0.

EPS = 1e-9   # epsilon kecil untuk hindari div/0

sdf_components = (sdf_with_prev_median
    # ── Price Gap ──────────────────────────────────────────────────────────
    .withColumn("raw_price_gap",
        F.when(
            (F.col("harga_baseline").isNull()) | (F.col("harga_baseline") <= 0) |
            (F.col("avg_harga").isNull()),
            F.lit(0.0)
        ).otherwise(
            (F.col("avg_harga") - F.col("harga_baseline")) / (F.col("harga_baseline") + F.lit(EPS))
        )
    )
    # ── Health Impact ───────────────────────────────────────────────────────
    .withColumn("raw_health_impact",
        F.when(
            (F.col("populasi_ternak").isNull()) | (F.col("populasi_ternak") <= 0),
            F.lit(0.0)
        ).otherwise(
            (F.col("sum_jumlah_sakit") + F.col("sum_jumlah_mati")) /
            (F.col("populasi_ternak") + F.lit(EPS))
        )
    )
    # ── Supply Strain ───────────────────────────────────────────────────────
    .withColumn("raw_supply_strain",
        F.when(
            (F.col("avg_permintaan_bulanan").isNull()) | (F.col("avg_permintaan_bulanan") <= 0),
            F.lit(0.0)
        ).otherwise(
            F.col("sum_vol_mutasi") / (F.col("avg_permintaan_bulanan") + F.lit(EPS))
        )
    )
)

print("✅ Komponen raw dihitung.")
sdf_components.select("id_wilayah","bulan","tahun","id_komoditas",
                       "raw_price_gap","raw_health_impact","raw_supply_strain").show(5)

✅ Komponen raw dihitung.
+----------+-----+-----+------------+-------------+--------------------+--------------------+
|id_wilayah|bulan|tahun|id_komoditas|raw_price_gap|   raw_health_impact|   raw_supply_strain|
+----------+-----+-----+------------+-------------+--------------------+--------------------+
|         1|    1| 2024|           1|          0.0|                 0.0|0.026346576670102043|
|         1|    2| 2024|           1|          0.0|8.234251452370255E-5|                 0.0|
|         1|    3| 2024|           1|          0.0|3.033671587715357...|                 0.0|
|         1|    4| 2024|           1|          0.0|                 0.0| 0.02012813000262454|
|         1|    5| 2024|           1|          0.0|                 0.0|0.017182550002240463|
+----------+-----+-----+------------+-------------+--------------------+--------------------+
only showing top 5 rows


In [28]:
# =========================================================
# CELL 27 · LANGKAH 5D-iii — MIN-MAX SCALING PER KOMPONEN
# =========================================================
# Scaling global: min & max dihitung dari seluruh dataset
# lalu dibroadcast. Menggunakan greatest/least untuk clamp ke [0,1].

def minmax_scale(df, col_raw: str, col_scaled: str):
    """Min-max scaling global, hasil di-clamp ke [0.0, 1.0]."""
    stats = df.agg(
        F.min(col_raw).alias("_min"),
        F.max(col_raw).alias("_max")
    ).collect()[0]
    v_min, v_max = float(stats["_min"] or 0), float(stats["_max"] or 1)
    denom = v_max - v_min if (v_max - v_min) != 0 else 1.0
    return (df
        .withColumn(col_scaled,
            F.greatest(
                F.lit(0.0),
                F.least(
                    F.lit(1.0),
                    (F.col(col_raw) - F.lit(v_min)) / F.lit(denom)
                )
            )
        )
    )

sdf_scaled = sdf_components
sdf_scaled = minmax_scale(sdf_scaled, "raw_price_gap",      "scaled_price_gap")
sdf_scaled = minmax_scale(sdf_scaled, "raw_health_impact",  "scaled_health_impact")
sdf_scaled = minmax_scale(sdf_scaled, "raw_supply_strain",  "scaled_supply_strain")

print("✅ Min-max scaling selesai.")
sdf_scaled.select("id_wilayah","bulan","tahun","id_komoditas",
                  "scaled_price_gap","scaled_health_impact","scaled_supply_strain").show(5)

✅ Min-max scaling selesai.
+----------+-----+-----+------------+----------------+--------------------+--------------------+
|id_wilayah|bulan|tahun|id_komoditas|scaled_price_gap|scaled_health_impact|scaled_supply_strain|
+----------+-----+-----+------------+----------------+--------------------+--------------------+
|         1|    1| 2024|           1|             0.0|                 0.0| 0.03454776457188917|
|         1|    2| 2024|           1|             0.0|0.003348595590631...|                 0.0|
|         1|    3| 2024|           1|             0.0|0.001233693112338...|                 0.0|
|         1|    4| 2024|           1|             0.0|                 0.0|0.026393633803368743|
|         1|    5| 2024|           1|             0.0|                 0.0|0.022531150807753808|
+----------+-----+-----+------------+----------------+--------------------+--------------------+
only showing top 5 rows


In [29]:
# =========================================================
# CELL 28 · LANGKAH 5D-iv — HITUNG supply_risk_index
# =========================================================
# Rata-rata sederhana 3 komponen (bobot sama 1/3 masing-masing)

sdf_with_index = (sdf_scaled
    .withColumn("supply_risk_index",
        F.round(
            (F.col("scaled_price_gap") +
             F.col("scaled_health_impact") +
             F.col("scaled_supply_strain")) / F.lit(3.0),
            6
        )
    )
)

print("✅ supply_risk_index dihitung.")
sdf_with_index.select("id_wilayah","bulan","tahun","id_komoditas","supply_risk_index").show(10)

✅ supply_risk_index dihitung.
+----------+-----+-----+------------+-----------------+
|id_wilayah|bulan|tahun|id_komoditas|supply_risk_index|
+----------+-----+-----+------------+-----------------+
|         1|    1| 2024|           1|         0.011516|
|         1|    2| 2024|           1|         0.001116|
|         1|    3| 2024|           1|          4.11E-4|
|         1|    4| 2024|           1|         0.008798|
|         1|    5| 2024|           1|          0.00751|
|         1|    2| 2025|           1|         0.001248|
|         1|    3| 2025|           1|         0.006664|
|         1|    5| 2025|           1|         0.005479|
|         1|    6| 2025|           1|         0.001335|
|         1|   10| 2025|           1|         0.007404|
+----------+-----+-----+------------+-----------------+
only showing top 10 rows


---
## Output — Tabel Fakta: `fact_supply_resilience`

Gabungkan semua surrogate key dari tabel dimensi ke dalam tabel fakta final.

In [30]:
# =========================================================
# CELL 29 · BANGUN fact_supply_resilience (INJEKSI SURROGATE KEY)
# =========================================================

sdf_fact_raw = (sdf_with_index.alias("f")
    # ── Join dim_prov → prov_key ──
    .join(sdf_dim_prov.alias("dp"),
          F.col("f.id_wilayah") == F.col("dp.id_wilayah"), how="left")
    # ── Join dim_komoditas → komoditas_key ──
    .join(sdf_dim_komoditas.alias("dk"),
          F.col("f.id_komoditas") == F.col("dk.id_komoditas"), how="left")
    # ── Join dim_waktu → waktu_key ──
    .join(sdf_dim_waktu.alias("dw"),
          (F.col("f.bulan") == F.col("dw.bulan")) &
          (F.col("f.tahun") == F.col("dw.tahun")),
          how="left")
    .select(
        # ── Surrogate Keys (FK ke dimensi) ──
        F.col("dp.prov_key"),
        F.col("dw.waktu_key"),
        F.col("dk.komoditas_key"),
        # ── Metrik iSIKHNAS ──
        F.col("f.sum_jumlah_sakit"),
        F.col("f.sum_jumlah_mati"),
        F.col("f.sum_vol_mutasi"),
        F.col("f.sum_realisasi_karkas"),
        # ── Metrik PIHPS ──
        F.round(F.col("f.avg_harga"), 2).alias("avg_harga"),
        # ── Metrik BPS (baseline & converted) ──
        F.round(F.col("f.harga_baseline"), 2).alias("harga_baseline"),
        F.round(F.col("f.populasi_ternak"), 0).alias("populasi_ternak"),
        F.round(F.col("f.avg_produksi_bulanan"), 2).alias("avg_produksi"),
        F.round(F.col("f.avg_konsumsi_bulanan"), 2).alias("avg_konsumsi"),
        F.round(F.col("f.avg_pemotongan_bulanan"), 2).alias("avg_pemotongan"),
        F.round(F.col("f.avg_permintaan_bulanan"), 2).alias("avg_permintaan"),
        # ── Indeks Risiko ──
        F.col("f.supply_risk_index"),
    )
    .filter(F.col("prov_key").isNotNull() & F.col("waktu_key").isNotNull())
)

sdf_fact_supply_resilience = sdf_fact_raw
sdf_fact_supply_resilience.cache()

total_rows = sdf_fact_supply_resilience.count()
print(f"✅ fact_supply_resilience siap: {total_rows} baris × {len(sdf_fact_supply_resilience.columns)} kolom")
sdf_fact_supply_resilience.printSchema()
sdf_fact_supply_resilience.show(10)

✅ fact_supply_resilience siap: 855 baris × 15 kolom
root
 |-- prov_key: integer (nullable = true)
 |-- waktu_key: integer (nullable = true)
 |-- komoditas_key: integer (nullable = true)
 |-- sum_jumlah_sakit: double (nullable = false)
 |-- sum_jumlah_mati: double (nullable = false)
 |-- sum_vol_mutasi: double (nullable = false)
 |-- sum_realisasi_karkas: double (nullable = false)
 |-- avg_harga: double (nullable = true)
 |-- harga_baseline: double (nullable = true)
 |-- populasi_ternak: double (nullable = true)
 |-- avg_produksi: double (nullable = true)
 |-- avg_konsumsi: double (nullable = true)
 |-- avg_pemotongan: double (nullable = true)
 |-- avg_permintaan: double (nullable = true)
 |-- supply_risk_index: double (nullable = true)

+--------+---------+-------------+----------------+---------------+--------------+--------------------+---------+--------------+---------------+------------+------------+--------------+--------------+-----------------+
|prov_key|waktu_key|komoditas_key|

---
## Validasi Hasil Transform

Cek distribusi, missing values, dan rentang indeks risiko sebelum dikirim ke tahap Load.

In [31]:
# =========================================================
# CELL 30 · VALIDASI HASIL TRANSFORM
# =========================================================

print("=" * 65)
print("  RINGKASAN TABEL DIMENSI")
print("=" * 65)
print(f"  dim_prov       : {sdf_dim_prov.count()} baris")
print(f"  dim_komoditas  : {sdf_dim_komoditas.count()} baris")
print(f"  dim_waktu      : {sdf_dim_waktu.count()} baris")

print("\n" + "=" * 65)
print("  RINGKASAN TABEL FAKTA")
print("=" * 65)
print(f"  fact_supply_resilience : {sdf_fact_supply_resilience.count()} baris")

print("\n── Missing Values pada Fact Table ──")
for col_name in sdf_fact_supply_resilience.columns:
    n_null = sdf_fact_supply_resilience.filter(F.col(col_name).isNull()).count()
    if n_null > 0:
        print(f"  ⚠️  {col_name:<30} → {n_null} null")
    else:
        print(f"  ✅ {col_name:<30} → 0 null")

print("\n── Statistik supply_risk_index ──")
sdf_fact_supply_resilience.select(
    F.min("supply_risk_index").alias("min"),
    F.max("supply_risk_index").alias("max"),
    F.avg("supply_risk_index").alias("mean"),
    F.stddev("supply_risk_index").alias("std"),
    F.percentile_approx("supply_risk_index", 0.5).alias("median"),
).show()

print("\n── Top 10 risiko tertinggi ──")
(sdf_fact_supply_resilience.alias("f")
    .join(sdf_dim_prov.alias("dp"),     "prov_key")
    .join(sdf_dim_komoditas.alias("dk"),"komoditas_key")
    .join(sdf_dim_waktu.alias("dw"),    "waktu_key")
    .select("dp.nama_provinsi","dw.tahun","dw.bulan","dk.nama_komoditas","f.supply_risk_index")
    .orderBy(F.col("f.supply_risk_index").desc())
    .show(10, truncate=False)
)

  RINGKASAN TABEL DIMENSI
  dim_prov       : 38 baris
  dim_komoditas  : 2 baris
  dim_waktu      : 72 baris

  RINGKASAN TABEL FAKTA
  fact_supply_resilience : 855 baris

── Missing Values pada Fact Table ──
  ✅ prov_key                       → 0 null
  ✅ waktu_key                      → 0 null
  ✅ komoditas_key                  → 0 null
  ✅ sum_jumlah_sakit               → 0 null
  ✅ sum_jumlah_mati                → 0 null
  ✅ sum_vol_mutasi                 → 0 null
  ✅ sum_realisasi_karkas           → 0 null
  ⚠️  avg_harga                      → 855 null
  ✅ harga_baseline                 → 0 null
  ✅ populasi_ternak                → 0 null
  ✅ avg_produksi                   → 0 null
  ✅ avg_konsumsi                   → 0 null
  ✅ avg_pemotongan                 → 0 null
  ✅ avg_permintaan                 → 0 null
  ✅ supply_risk_index              → 0 null

── Statistik supply_risk_index ──
+---+--------+--------------------+-------------------+--------+
|min|     max|             

In [32]:
# =========================================================
# CELL 31 · SIMPAN SEMUA DATAFRAME TRANSFORM SEBAGAI PARQUET
#            (Checkpoint sebelum Fase Load)
# =========================================================
import os

OUTPUT_DIR = "../../DATA/TRANSFORM_OUTPUT"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tables_to_save = {
    "dim_prov":                sdf_dim_prov,
    "dim_komoditas":           sdf_dim_komoditas,
    "dim_waktu":               sdf_dim_waktu,
    "ref_wilayah_master":      sdf_wilayah_master,
    "ref_komoditas":           sdf_ref_komoditas,
    "tr_demografi":            sdf_tr_demografi,
    "tr_statistik":            sdf_tr_statistik_imputed,
    "fact_supply_resilience":  sdf_fact_supply_resilience,
}

print("💾 Menyimpan output transform ke Parquet...\n")
for name, df in tables_to_save.items():
    path = os.path.join(OUTPUT_DIR, name)
    df.write.mode("overwrite").parquet(path)
    print(f"  ✅ {name:<30} → {path}")

print("\n✅ Semua tabel transform tersimpan sebagai Parquet.")
print("   Siap dilanjutkan ke Fase 3: LOAD → Data Warehouse.")
print(f"   Output directory: {os.path.abspath(OUTPUT_DIR)}")

💾 Menyimpan output transform ke Parquet...



Py4JJavaError: An error occurred while calling o2024.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

---
## Ringkasan Fase Transform

| Langkah | Proses | Output |
|---|---|---|
| **1** | Cleaning & Merge BPS (API + Dummy) | `sdf_bps_master` |
| **2** | Normalisasi & Unpivot BPS | `ref_wilayah`, `ref_komoditas`, `tr_demografi`, `tr_statistik` |
| **3** | Standardisasi provinsi, Harmonisasi tanggal, Imputasi | Data bersih per sumber |
| **4** | Bangun dimensi (surrogate key) | `dim_prov`, `dim_komoditas`, `dim_waktu` |
| **5A** | Agregasi iSIKHNAS & PIHPS per (wilayah, bulan, komoditas) | Metrik operasional |
| **5B** | Konversi satuan & normalisasi waktu BPS | Metrik bulanan BPS |
| **5C** | Integrasi INNER JOIN agregasi + BPS | Dataset terintegrasi |
| **5D** | Hitung `supply_risk_index` (Price Gap + Health Impact + Supply Strain) | `fact_supply_resilience` |

**Tabel siap Load ke Data Warehouse:**
- `dim_prov` (38 baris)
- `dim_komoditas` (2 baris)
- `dim_waktu` (deret bulan otomatis)
- `fact_supply_resilience` (tabel fakta utama)

> **Next step → Fase 3: LOAD** — Load semua tabel di atas ke PostgreSQL `datawarehouse_db` menggunakan PySpark JDBC writer.
